In [2]:
from memoria_icm.datos import *
from memoria_icm.funciones import *
from memoria_icm.graficos import *
import pandas as pd

In [3]:
df, df_resumen = cargar_datos()#estos son los definitivos

# import sqlite3
# ruta = os.path.join(BASE_DIR, "data", "data_raw.db")

# con = sqlite3.connect(ruta)

# df = pd.read_sql('SELECT * FROM datos_trackeo', con=con)
# df_resumen = pd.read_sql('SELECT * FROM datos_resumen', con)

Datos cargados. Trackeo: (44324, 28), Resumen: (192, 42)


In [93]:
# df_resumen = df_resumen[df_resumen['video_analizado'] == 1]
# df_resumen

In [4]:
columnas = list(df_resumen.select_dtypes(include='float64').columns)
columnas

['duracion_video',
 'r_promedio',
 'r_mediana',
 'r_centrado_promedio',
 'r_centrado_mediana',
 'v_promedio',
 'v_mediana',
 'a_promedio',
 'a_mediana',
 'dr_promedio',
 'dr_mediana',
 'dr_centrado_promedio',
 'dr_centrado_mediana',
 'theta_r_promedio',
 'theta_r_mediana',
 'theta_r_centrado_promedio',
 'theta_r_centrado_mediana',
 'v_angular_promedio',
 'v_angular_mediana',
 'v_angular_centrado_promedio',
 'v_angular_centrado_mediana',
 'a_angular_promedio',
 'a_angular_mediana',
 'a_angular_centrado_promedio',
 'a_angular_centrado_mediana',
 'curvatura_promedio',
 'curvatura_mediana',
 'theta_v_promedio',
 'theta_v_mediana']

In [5]:
erizos_e= df_resumen[(df_resumen['especie'] == 'LA') & (df_resumen['tratamiento'] == 'E')]
erizos_r = df_resumen[(df_resumen['especie'] == 'LA') & (df_resumen['tratamiento'] == 'R')] 
locos_e = df_resumen[(df_resumen['especie'] == 'CC') & (df_resumen['tratamiento'] == 'E')]
locos_r = df_resumen[(df_resumen['especie'] == 'CC') & (df_resumen['tratamiento'] == 'R')]  

In [6]:
grupos = {'erizos_e': erizos_e, 'erizos_r':erizos_r, 'locos_e':locos_e, 'locos_r':locos_r}

In [91]:
erizos_e

,level_0,index,id,id_ubicacion,especie,tratamiento,salinidad,nombre_archivo,estadio_larval,obs,...,v_angular_centrado_mediana,a_angular_promedio,a_angular_mediana,a_angular_centrado_promedio,a_angular_centrado_mediana,curvatura_promedio,curvatura_mediana,theta_v_promedio,theta_v_mediana,nivel_salinidad
67,72,72,84,1,LA,E,35-N,Video_001,Larva equinopluteus,1,...,0.085948,-0.022731,0.016664,5.842938,0.029081,131.115034,86.589162,-0.825388,-1.099906,normal
68,73,73,86,1,LA,E,35-N,Video_003,Larva equinopluteus,0,...,0.093822,-0.023083,0.018873,-2.916230,0.234377,130.452264,103.227284,-0.194273,-0.451203,normal
69,75,75,88,1,LA,E,35-N,Video_005,Larva equinopluteus,1,...,-0.144786,0.005172,0.007212,4.023243,-0.066822,62.893341,49.618214,-0.128535,-0.684617,normal
70,77,77,91,1,LA,E,35-R,Video_002,Larva equinopluteus,1,...,0.368764,-0.064507,-0.000810,-5.065409,-0.335211,55.198759,21.939539,-0.014635,0.231887,normal
71,78,78,92,1,LA,E,35-R,Video_003,Larva equinopluteus,0,...,0.179788,-0.000555,-0.015739,-1.028561,-0.019221,57.567456,32.275352,1.232831,1.154498,normal
72,80,80,94,1,LA,E,35-R,Video_005,Larva equinopluteus,1,...,0.033834,0.039527,-0.016947,5.498720,0.057394,109.204802,78.170738,-0.394800,-1.144946,normal
73,81,81,95,1,LA,E,35-R,Video_006,Larva equinopluteus,0,...,0.138801,0.009783,-0.006693,-0.265606,-0.047338,111.883159,71.439835,-0.827983,-1.105241,normal
74,83,83,97,1,LA,E,40,Video_002,Larva equinopluteus,1,...,0.155009,0.049210,0.055945,0.970934,-0.284862,321.131014,209.948917,-0.508248,-1.002167,alta
75,84,84,99,1,LA,E,40,Video_004,Larva equinopluteus,0,...,0.018530,0.011232,0.010038,1.317069,-0.004861,131.513946,83.361683,0.015912,0.034007,alta
76,86,86,101,1,LA,E,40,Video_006,Larva equinopluteus,1,...,-0.038469,0.052230,0.005099,3.855016,-0.264024,202.372266,164.143090,-0.286278,-1.160159,alta


### ESTADIO LARVAL

In [86]:
resultados = {}
factor = 'estadio_larval'
variables = [var for var in variables_resumen if var in df_resumen.columns]

for nombre_grupo, df in grupos.items():
    clave = f"{nombre_grupo}"
    resultados[clave] = pd.concat([testear_supuestos(df, var, factor) for var in variables],
                                  ignore_index=True)
    
df_resultados = pd.concat([lista.assign(grupo=clave) for clave, lista in resultados.items()],
                          ignore_index=True)

resultados_erizos_r = df_resultados[(df_resultados['grupo'] == 'erizos_e')]

In [87]:
variables_resumen_por_test = (resultados_erizos_r[['variable', 'test_sugerido']].drop_duplicates()
    .groupby('test_sugerido')['variable']
    .unique()
    .to_dict())

print("ERIZOS - QUÉ APLICAR SEGÚN SUPUESTOS")
for tipo, lista in variables_resumen_por_test.items():
    print(f"Puedo aplicar {tipo} en: {list(lista)}")
    
listas_por_test = {
    tipo: grupo['variable'].drop_duplicates().tolist()
    for tipo, grupo in resultados_erizos_r.dropna(subset=['test_sugerido']).groupby('test_sugerido')}

ERIZOS - QUÉ APLICAR SEGÚN SUPUESTOS
Puedo aplicar ANOVA en: ['theta_r_centrado_promedio', 'theta_r_centrado_mediana', 'v_angular_centrado_promedio', 'theta_v_mediana', 'theta_v_promedio']
Puedo aplicar Kruskal-Wallis en: ['dr_centrado_promedio', 'theta_r_mediana', 'curvatura_mediana', 'a_angular_centrado_promedio', 'v_mediana', 'a_promedio', 'a_angular_centrado_mediana', 'a_angular_promedio', 'r_centrado_promedio', 'dr_promedio', 'a_angular_mediana', 'a_mediana', 'r_centrado_mediana', 'v_angular_promedio', 'dr_mediana', 'theta_r_promedio', 'r_mediana', 'v_angular_mediana', 'v_angular_centrado_mediana', 'curvatura_promedio', 'dr_centrado_mediana', 'v_promedio', 'r_promedio']


In [ ]:
# df_efectos_erizos = analizar_grupo(erizos_e, listas_por_test, factor)

# pvalues_significativos_erizos = df_efectos_erizos.loc[(df_efectos_erizos['p_value'] < 0.05)]
# variables_candidatas_erizos = set(pvalues_significativos_erizos['variable'].tolist())

In [ ]:
# print(df_efectos_erizos)

              variable            test       p_value        η²     ω²/ε²
1    curvatura_mediana  Kruskal-Wallis  2.149548e-07  0.358583  0.349915
14  curvatura_promedio  Kruskal-Wallis  1.016773e-06  0.318615  0.309407
2            v_mediana  Kruskal-Wallis  4.645656e-06  0.279708  0.269974
15          v_promedio  Kruskal-Wallis  5.141468e-06  0.277118  0.267349
13   v_angular_mediana  Kruskal-Wallis  3.419437e-04  0.171005  0.159803
7            a_mediana  Kruskal-Wallis  3.134074e-03  0.116370  0.104429
3           a_promedio  Kruskal-Wallis  3.356839e-03  0.114701  0.102738
8   v_angular_promedio  Kruskal-Wallis  1.831601e-02  0.074209  0.061698
11    theta_r_promedio  Kruskal-Wallis  7.082945e-02  0.043516  0.030590
0      theta_r_mediana  Kruskal-Wallis  9.314292e-02  0.037589  0.024584
12           r_mediana  Kruskal-Wallis  1.340320e-01  0.029936  0.016827
17          r_promedio  Kruskal-Wallis  1.513930e-01  0.027442  0.014299
6    a_angular_mediana  Kruskal-Wallis  1.984946e-0

### OBS Y LARVA ÚNICA (reemplazar)

In [95]:
df_resumen = df_resumenog

In [7]:
df_resumen

,level_0,index,id,id_ubicacion,especie,tratamiento,salinidad,nombre_archivo,estadio_larval,obs,...,v_angular_centrado_mediana,a_angular_promedio,a_angular_mediana,a_angular_centrado_promedio,a_angular_centrado_mediana,curvatura_promedio,curvatura_mediana,theta_v_promedio,theta_v_mediana,nivel_salinidad
0,0,0,0,1,CC,E,35-N,Video_001,Larva recién eclosionada,0,...,0.341320,-0.012305,0.000266,1.166878,-0.107830,86.111707,67.287227,0.446752,0.505336,normal
1,1,1,1,1,CC,E,35-N,Video_002,Larva recién eclosionada,0,...,0.130120,0.013244,0.015889,-3.061171,-0.102260,238.659192,94.229832,0.015796,-0.120120,normal
2,2,2,3,1,CC,E,35-N,Video_004,Larva recién eclosionada,0,...,0.025293,0.023416,-0.006303,-3.860114,-0.027329,229.644998,183.808560,0.478789,0.926164,normal
3,3,3,4,1,CC,E,35-N,Video_005,Larva recién eclosionada,1,...,-1.239164,-0.013387,-0.006991,4.159481,0.391242,211.802054,142.603931,0.210658,0.276620,normal
4,4,4,5,1,CC,E,35-N,Video_006,Larva recién eclosionada,0,...,0.013452,-0.023944,-0.023028,-3.250676,-0.033807,166.512069,36.641630,-0.719150,-1.145635,normal
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
187,261,261,316,3,LA,R,35,Video_002,Larva equinopluteus,0,...,0.001953,-0.004636,0.007926,0.805636,0.019622,36.039624,23.368477,2.067259,2.437282,normal
188,262,262,317,3,LA,R,35,Video_003,Larva equinopluteus,0,...,-0.021381,0.001006,-0.001965,-6.663943,-0.000412,112.075803,86.970219,1.743853,2.130771,normal
189,263,263,318,3,LA,R,35,Video_004,Larva equinopluteus,0,...,0.032448,0.011702,0.004621,2.595045,0.010800,23.874776,14.785156,-0.467722,-0.365288,normal
190,264,264,319,3,LA,R,35,Video_005,Larva equinopluteus,0,...,-0.159212,0.002101,0.000895,5.702312,0.143047,85.503557,49.689911,0.482970,0.186716,normal


In [8]:
resultados = {}
factor = 'larva_unica' #reemplazar
variables = [var for var in variables_resumen if var in columnas]

for nombre_grupo, df in grupos.items():
    clave = f"{nombre_grupo}"
    resultados[clave] = pd.concat([testear_supuestos(df, var, factor) for var in variables],
                                  ignore_index=True)
    
df_resultados = pd.concat([lista.assign(grupo=clave) for clave, lista in resultados.items()],
                          ignore_index=True)

obs = df_resultados[df_resultados['grupo'] == 'locos_r']

In [9]:
variables_resumen_por_test = (obs[['variable', 'test_sugerido']].drop_duplicates()
    .groupby('test_sugerido')['variable']
    .unique()
    .to_dict())

print("QUÉ APLICAR SEGÚN SUPUESTOS")
for tipo, lista in variables_resumen_por_test.items():
    print(f"Puedo aplicar {tipo} en: {list(lista)}")
    
listas_por_test = {
    tipo: grupo['variable'].drop_duplicates().tolist()
    for tipo, grupo in obs.dropna(subset=['test_sugerido']).groupby('test_sugerido')}

QUÉ APLICAR SEGÚN SUPUESTOS
Puedo aplicar ANOVA en: ['theta_r_centrado_promedio', 'theta_r_centrado_mediana', 'theta_v_mediana']
Puedo aplicar Kruskal-Wallis en: ['theta_r_promedio', 'v_angular_centrado_mediana', 'a_promedio', 'a_angular_centrado_promedio', 'a_angular_centrado_mediana', 'r_centrado_mediana', 'a_angular_mediana', 'v_angular_mediana', 'r_centrado_promedio', 'r_mediana', 'dr_mediana', 'theta_v_promedio', 'dr_promedio', 'a_angular_promedio', 'a_mediana', 'curvatura_mediana', 'v_angular_centrado_promedio', 'dr_centrado_promedio', 'v_angular_promedio', 'v_promedio', 'theta_r_mediana', 'curvatura_promedio', 'v_mediana', 'r_promedio', 'dr_centrado_mediana']


In [10]:
df_efectos = analizar_grupo(locos_r, listas_por_test, factor)

pvalues_significativos = df_efectos.loc[(df_efectos['p_value'] < 0.05)]
print(pvalues_significativos)

                variable            test   p_value        η²     ω²/ε²
15     curvatura_mediana  Kruskal-Wallis  0.000078  0.273768  0.260799
5     r_centrado_mediana  Kruskal-Wallis  0.000141  0.254277  0.240961
22             v_mediana  Kruskal-Wallis  0.000155  0.251099  0.237726
19            v_promedio  Kruskal-Wallis  0.000155  0.251099  0.237726
8    r_centrado_promedio  Kruskal-Wallis  0.000206  0.241683  0.228142
21    curvatura_promedio  Kruskal-Wallis  0.000206  0.241683  0.228142
14             a_mediana  Kruskal-Wallis  0.000515  0.211598  0.197520
2             a_promedio  Kruskal-Wallis  0.000515  0.211598  0.197520
17  dr_centrado_promedio  Kruskal-Wallis  0.002161  0.165041  0.150131
23            r_promedio  Kruskal-Wallis  0.006276  0.131038  0.115521
24   dr_centrado_mediana  Kruskal-Wallis  0.007782  0.124261  0.108623
9              r_mediana  Kruskal-Wallis  0.007782  0.124261  0.108623


In [ ]:
df_resumen

,level_0,index,id,id_ubicacion,especie,tratamiento,salinidad,nombre_archivo,estadio_larval,obs,...,v_angular_centrado_mediana,a_angular_promedio,a_angular_mediana,a_angular_centrado_promedio,a_angular_centrado_mediana,curvatura_promedio,curvatura_mediana,theta_v_promedio,theta_v_mediana,nivel_salinidad
0,0,0,0,1,CC,E,35-N,Video_001,Larva recién eclosionada,0,...,0.341320,-0.012305,0.000266,1.166878,-0.107830,86.111707,67.287227,0.446752,0.505336,normal
1,1,1,1,1,CC,E,35-N,Video_002,Larva recién eclosionada,0,...,0.130120,0.013244,0.015889,-3.061171,-0.102260,238.659192,94.229832,0.015796,-0.120120,normal
2,2,2,3,1,CC,E,35-N,Video_004,Larva recién eclosionada,0,...,0.025293,0.023416,-0.006303,-3.860114,-0.027329,229.644998,183.808560,0.478789,0.926164,normal
3,3,3,4,1,CC,E,35-N,Video_005,Larva recién eclosionada,1,...,-1.239164,-0.013387,-0.006991,4.159481,0.391242,211.802054,142.603931,0.210658,0.276620,normal
4,4,4,5,1,CC,E,35-N,Video_006,Larva recién eclosionada,0,...,0.013452,-0.023944,-0.023028,-3.250676,-0.033807,166.512069,36.641630,-0.719150,-1.145635,normal
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
187,261,261,316,3,LA,R,35,Video_002,Larva equinopluteus,0,...,0.001953,-0.004636,0.007926,0.805636,0.019622,36.039624,23.368477,2.067259,2.437282,normal
188,262,262,317,3,LA,R,35,Video_003,Larva equinopluteus,0,...,-0.021381,0.001006,-0.001965,-6.663943,-0.000412,112.075803,86.970219,1.743853,2.130771,normal
189,263,263,318,3,LA,R,35,Video_004,Larva equinopluteus,0,...,0.032448,0.011702,0.004621,2.595045,0.010800,23.874776,14.785156,-0.467722,-0.365288,normal
190,264,264,319,3,LA,R,35,Video_005,Larva equinopluteus,0,...,-0.159212,0.002101,0.000895,5.702312,0.143047,85.503557,49.689911,0.482970,0.186716,normal
